# Semantic Caching [Step 1 - Cache by Semantic Similarity]

> **MLCourse - Agentic AI - Cache RAG**

In RAG pipelines, many queries are semantically similar even if worded
differently. Semantic caching stores embeddings alongside answers and
retrieves cached results when a new query is close enough in embedding
space. This notebook builds a cache that computes cosine similarity
between incoming queries and stored entries, returning cached answers
when a similarity threshold is exceeded.

In [1]:
import os
import time
import json
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


### 1. Load and Chunk the Document


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print(f"Loaded alice.txt: {len(raw_text)} chars -> {len(chunks)} chunks")


### 2. Build the Vector Store for Retrieval


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_cache_rag")
print(f"Vector store built with {vectorstore._collection.count()} vectors")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready.")


### 3. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)


### 4. Define the Semantic Cache


In [ ]:
# The cache stores entries as dictionaries with three fields:
# - query: the original user query string
# - embedding: the vector representation of the query
# - answer: the generated answer for that query
# - timestamp: when the entry was created
#
# Cosine similarity is computed between the new query embedding and
# each cached embedding. If the max similarity exceeds the threshold,
# we return the cached answer instead of regenerating.

import numpy as np

class SemanticCache:
    """A simple in-memory semantic cache using cosine similarity."""

    def __init__(self, embeddings_model, threshold=0.85):
        self.embeddings_model = embeddings_model
        self.threshold = threshold
        self.cache = []  # list of dicts: {query, embedding, answer, timestamp}

    def _get_embedding(self, text):
        """Compute embedding for a text string."""
        return self.embeddings_model.embed_query(text)

    def _cosine_similarity(self, a, b):
        """Compute cosine similarity between two vectors."""
        a = np.array(a)
        b = np.array(b)
        dot = np.dot(a, b)
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return float(dot / (norm_a * norm_b))

    def lookup(self, query):
        """Check if a semantically similar query exists in cache."""
        if not self.cache:
            return None, 0.0
        query_emb = self._get_embedding(query)
        best_score = -1.0
        best_entry = None
        for entry in self.cache:
            score = self._cosine_similarity(query_emb, entry["embedding"])
            if score > best_score:
                best_score = score
                best_entry = entry
        if best_score >= self.threshold:
            return best_entry, best_score
        return None, best_score

    def store(self, query, answer):
        """Store a new query-answer pair in the cache."""
        embedding = self._get_embedding(query)
        self.cache.append({
            "query": query,
            "embedding": embedding,
            "answer": answer,
            "timestamp": time.time(),
        })

    def size(self):
        return len(self.cache)

    def clear(self):
        self.cache = []

print("SemanticCache class defined.")
print("Threshold: 0.85 (cosine similarity)")


### 5. Initialize the Cache


In [ ]:
cache = SemanticCache(embeddings, threshold=0.85)
print(f"Cache initialized with threshold={cache.threshold}")


### 6. Build the Cached Query Function


In [ ]:
# This function first checks the cache. On a hit, it returns immediately.
# On a miss, it retrieves relevant documents, generates an answer with the
# LLM, and stores the result in the cache for future queries.

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question using ONLY the provided context. "
     "If the context does not contain enough information, say so. "
     "Be concise and accurate."),
    ("user", "Context:\n{context}\n\nQuestion: {query}")
])

def cached_query(query, verbose=True):
    """Query with semantic caching. Returns (answer, from_cache, similarity)."""
    # Step 1: Check cache
    hit, score = cache.lookup(query)
    if hit is not None:
        if verbose:
            print(f"  [CACHE HIT] sim={score:.4f} -- returning cached answer")
        return hit["answer"], True, score

    if verbose:
        print(f"  [CACHE MISS] best_sim={score:.4f} -- generating new answer")

    # Step 2: Retrieve documents
    docs = retriever.invoke(query)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    if verbose:
        print(f"  [RETRIEVE] Got {len(docs)} docs ({len(context)} chars)")

    # Step 3: Generate answer
    response = (rag_prompt | llm).invoke({"context": context, "query": query})
    answer = response.content
    if verbose:
        print(f"  [GENERATE] {answer[:80]}...")

    # Step 4: Store in cache
    cache.store(query, answer)
    if verbose:
        print(f"  [STORE] Cache size: {cache.size()}")

    return answer, False, 0.0

print("cached_query function defined.")


### 7. Test: Initial Query (Cache Miss)


In [ ]:
print("=" * 60)
print("TEST 1: First query -- should be a cache miss")
print("=" * 60)
answer, from_cache, sim = cached_query("Who does Alice meet at the tea party?")
print(f"\nAnswer: {answer[:200]}")
print(f"From cache: {from_cache}")


### 8. Test: Exact Duplicate (Cache Hit)


In [ ]:
print("\n" + "=" * 60)
print("TEST 2: Exact same query -- should be a cache hit")
print("=" * 60)
answer, from_cache, sim = cached_query("Who does Alice meet at the tea party?")
print(f"\nAnswer: {answer[:200]}")
print(f"From cache: {from_cache}, similarity: {sim:.4f}")


### 9. Test: Semantically Similar Query (Should Hit Cache)


In [ ]:
print("\n" + "=" * 60)
print("TEST 3: Similar phrasing -- should be a cache hit")
print("=" * 60)
answer, from_cache, sim = cached_query(
    "Which characters does Alice encounter at the Mad Hatter's tea party?"
)
print(f"\nAnswer: {answer[:200]}")
print(f"From cache: {from_cache}, similarity: {sim:.4f}")


### 10. Test: Another Similar Variant


In [ ]:
print("\n" + "=" * 60)
print("TEST 4: Another variant -- should hit cache")
print("=" * 60)
answer, from_cache, sim = cached_query(
    "Tell me about the tea party scene and who is there"
)
print(f"\nAnswer: {answer[:200]}")
print(f"From cache: {from_cache}, similarity: {sim:.4f}")


### 11. Test: Different Topic (Should Miss Cache)


In [ ]:
print("\n" + "=" * 60)
print("TEST 5: Different topic -- should be a cache miss")
print("=" * 60)
answer, from_cache, sim = cached_query("What is the Queen of Hearts' punishment?")
print(f"\nAnswer: {answer[:200]}")
print(f"From cache: {from_cache}")


### 12. Test: Yet Another Different Topic


In [ ]:
print("\n" + "=" * 60)
print("TEST 6: Another new topic -- cache miss")
print("=" * 60)
answer, from_cache, sim = cached_query("What games does the Queen of Hearts play?")
print(f"\nAnswer: {answer[:200]}")
print(f"From cache: {from_cache}")


### 13. Inspect Cache Contents


In [ ]:
print("\n" + "=" * 60)
print("CACHE STATE")
print("=" * 60)
print(f"Total entries: {cache.size()}")
for i, entry in enumerate(cache.cache):
    print(f"  [{i}] query: '{entry['query'][:60]}...'")
    print(f"       answer: '{entry['answer'][:60]}...'")


### 14. Threshold Experiment


In [ ]:
# Let us see what happens when we tighten the threshold. A higher threshold
# means fewer false-positive hits but more cache misses.

strict_cache = SemanticCache(embeddings, threshold=0.95)
print("Strict cache initialized (threshold=0.95)")

# First query to populate
answer, _, _ = cached_query("What happened when Alice fell down the rabbit hole?", verbose=False)
strict_cache.store("What happened when Alice fell down the rabbit hole?", answer)

# Test with a similar query
test_query = "Tell me about Alice falling into the rabbit hole"
query_emb = embeddings.embed_query(test_query)
cached_emb = strict_cache.cache[0]["embedding"]
score = strict_cache._cosine_similarity(query_emb, cached_emb)
print(f"\nSimilarity between similar queries: {score:.4f}")
print(f"Strict threshold (0.95): {'HIT' if score >= 0.95 else 'MISS'}")
print(f"Normal threshold (0.85): {'HIT' if score >= 0.85 else 'MISS'}")

# Test with a less similar query
test_query2 = "What is Alice's name?"
query_emb2 = embeddings.embed_query(test_query2)
score2 = strict_cache._cosine_similarity(query_emb2, cached_emb)
print(f"\nSimilarity for different query: {score2:.4f}")
print(f"Strict threshold (0.95): {'HIT' if score2 >= 0.95 else 'MISS'}")
print(f"Normal threshold (0.85): {'HIT' if score2 >= 0.85 else 'MISS'}")


### 15. Cache Performance Summary


In [ ]:
print("\n" + "=" * 60)
print("PERFORMANCE SUMMARY")
print("=" * 60)
print(f"Cache size: {cache.size()} entries")
print(f"Threshold: {cache.threshold}")
print("Benefits of semantic caching:")
print("  1. Exact duplicate queries: instant return, zero LLM calls")
print("  2. Similar phrasing: fast return if above similarity threshold")
print("  3. Novel queries: normal RAG pipeline, result cached for later")
print("  4. Reduces latency: avoids document retrieval + LLM generation")
print("  5. Reduces cost: fewer embedding calls, fewer LLM tokens")


### Summary
